In [0]:
## 1. Carga del Dataset Limpio

df_analisis = spark.table("superstore_limpio")
display(df_analisis)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
26,CA-2017-121755,2017-01-16,2017-01-20,Second Class,EH-13945,Eric Hoffmann,Consumer,United States,Los Angeles,California,90049,West,OFF-BI-10001634,Office Supplies,Binders,Wilson Jones Active Use Binders,11.648,2,0.2,4.2224
29,US-2016-150630,2016-09-17,2016-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,OFF-BI-10000474,Office Supplies,Binders,Avery Recycled Flexi-View Covers for Binding Systems,9.618,2,0.7,-7.0532
30,US-2016-150630,2016-09-17,2016-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,FUR-FU-10004848,Furniture,Furnishings,"Howard Miller 13-3/4"" Diameter Brushed Chrome Round Wall Clock",124.2,3,0.2,15.525
57,CA-2017-111682,2017-06-17,2017-06-18,First Class,TB-21055,Ted Butterfield,Consumer,United States,Troy,New York,12180,East,OFF-PA-10001569,Office Supplies,Paper,Xerox 232,32.4,5,0.0,15.552
74,US-2016-134026,2016-04-26,2016-05-02,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,FUR-FU-10003708,Furniture,Furnishings,"Tenex Traditional Chairmats for Medium Pile Carpet, Standard Lip, 36"" x 48""",97.04,2,0.2,1.213
75,US-2016-134026,2016-04-26,2016-05-02,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,OFF-ST-10004123,Office Supplies,Storage,Safco Industrial Wire Shelving System,72.784,1,0.2,-18.196
82,CA-2015-139451,2015-10-12,2015-10-16,Standard Class,DN-13690,Duane Noonan,Consumer,United States,San Francisco,California,94122,West,OFF-AR-10002053,Office Supplies,Art,"Premium Writing Pencils, Soft, #2 by Central Association for the Blind",14.9,5,0.0,4.172
92,CA-2017-109806,2017-09-17,2017-09-22,Standard Class,JS-15685,Jim Sink,Corporate,United States,Los Angeles,California,90036,West,OFF-PA-10000304,Office Supplies,Paper,Xerox 1995,6.48,1,0.0,3.1104
118,CA-2016-110457,2016-03-02,2016-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0.0,165.3813
128,US-2018-107272,2018-11-05,2018-11-12,Standard Class,TS-21610,Troy Staebel,Consumer,United States,Phoenix,Arizona,85023,West,OFF-ST-10002974,Office Supplies,Storage,"Trav-L-File Heavy-Duty Shuttle II, Black",243.992,7,0.2,30.499


In [0]:
## 2. Filtros del Dashboard

dbutils.widgets.dropdown("region_filtro", "Todas", ["Todas", "West", "East", "Central", "South"], "Región")
dbutils.widgets.dropdown("categoria_filtro", "Todas", ["Todas", "Furniture", "Office Supplies", "Technology"], "Categoría")

region_sel = dbutils.widgets.get("region_filtro")
categoria_sel = dbutils.widgets.get("categoria_filtro")

df_filtrado = df_analisis
if region_sel != "Todas":
    df_filtrado = df_filtrado.filter(df_filtrado.Region == region_sel)
if categoria_sel != "Todas":
    df_filtrado = df_filtrado.filter(df_filtrado.Category == categoria_sel)

print(f"Región seleccionada: {region_sel} | Categoría seleccionada: {categoria_sel}")
print(f"Registros después de filtrar: {df_filtrado.count()}")

Región seleccionada: Todas | Categoría seleccionada: Todas
Registros después de filtrar: 9983


In [0]:
## 3. Indicadores Clave (KPIs)

import pyspark.sql.functions as F

kpis = df_filtrado.agg(
    F.round(F.sum("Sales"), 2).alias("Ventas_Totales"),
    F.round(F.sum("Profit"), 2).alias("Ganancia_Total"),
    F.countDistinct("Order_ID").alias("Total_Pedidos")
)
display(kpis)

Ventas_Totales,Ganancia_Total,Total_Pedidos
2288271.49,284152.04,5003


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
## 4. Gráfico 1: Ventas por Región

ventas_region = df_filtrado.groupBy("Region") \
                           .agg(F.round(F.sum("Sales"), 2).alias("Ventas_Totales")) \
                           .orderBy(F.desc("Ventas_Totales"))
display(ventas_region)

Region,Ventas_Totales
West,725457.82
East,669851.87
Central,501239.89
South,391721.91


Databricks visualization. Run in Databricks to view.

In [0]:
## 5. Gráfico 2: Top 10 Productos con Mayor Ganancia

top_productos = df_filtrado.groupBy("Product_Name") \
                           .agg(F.round(F.sum("Profit"), 2).alias("Ganancia_Total")) \
                           .orderBy(F.desc("Ganancia_Total")) \
                           .limit(10)
display(top_productos)

Product_Name,Ganancia_Total
Canon imageCLASS 2200 Advanced Copier,25199.93
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,7753.04
Hewlett Packard LaserJet 3310 Copier,6983.88
Canon PC1060 Personal Laser Copier,4570.93
"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",4094.98
Ativa V4110MDD Micro-Cut Shredder,3772.95
"3D Systems Cube Printer, 2nd Generation, Magenta",3717.97
Plantronics Savi W720 Multi-Device Wireless Headset System,3696.28
Ibico EPK-21 Electric Binding System,3345.28
Zebra ZM400 Thermal Label Printer,3343.54


Databricks visualization. Run in Databricks to view.

In [0]:
## 6. Gráfico 3: Rendimiento por Categoría

ventas_categoria = df_filtrado.groupBy("Category") \
                              .agg(F.round(F.sum("Sales"), 2).alias("Ventas_Totales")) \
                              .orderBy(F.desc("Ventas_Totales"))
display(ventas_categoria)

Category,Ventas_Totales
Technology,834554.27
Furniture,736879.7
Office Supplies,716837.52


Databricks visualization. Run in Databricks to view.

In [0]:
## 7. Gráfico 4: Distribución de Ventas y Ganancia

display(df_filtrado.select("Sales", "Profit").describe())

display(df_filtrado.select("Sales", "Profit"))

summary,Sales,Profit
count,9983,9983
mean,229.21681761994978,28.463592447160107
stddev,621.909609883406,234.1221067198018
min,0.444,-6599.978
max,22638.48,8399.976


Sales,Profit
11.648,4.2224
9.618,-7.0532
124.2,15.525
32.4,15.552
97.04,1.213
72.784,-18.196
14.9,4.172
6.48,3.1104
787.53,165.3813
243.992,30.499


Databricks visualization. Run in Databricks to view.

In [0]:
## 8. Gráfico 5: Relación entre Descuento y Ganancia

correlacion = df_filtrado.stat.corr("Discount", "Profit")
print(f"Correlación entre Discount y Profit: {correlacion:.3f}")

display(df_filtrado.select("Discount", "Profit"))

Correlación entre Discount y Profit: -0.219


Discount,Profit
0.2,4.2224
0.7,-7.0532
0.2,15.525
0.0,15.552
0.2,1.213
0.2,-18.196
0.0,4.172
0.0,3.1104
0.0,165.3813
0.2,30.499


Databricks visualization. Run in Databricks to view.

### Interpretación de los Resultados
Este dashboard resume el comportamiento de ventas y rentabilidad de la tienda 
Superstore, con base en los 9,983 pedidos ya depurados.

Con los filtros de Región y Categoría en "Todas", la tienda generó 2,288,271 en 
ventas totales y 284,152 en ganancia, a partir de 5,003 pedidos únicos.

La región West concentra la mayor proporción de ventas (31.7%), seguida de East 
(29.3%), Central (21.9%) y South (17.1%), lo que sugiere priorizar inventario y 
campañas comerciales en la costa oeste.

El producto más rentable es la Canon imageCLASS 2200 Advanced Copier, con una 
ganancia acumulada muy por encima del resto del top 10, seguida de otros 
equipos de oficina como el Hewlett Packard LaserJet 3310 y el Fellowes PB500 
Electric Punch. Esto indica que los equipos de impresión y copiado de gama 
media-alta son los productos más estratégicos para la rentabilidad del negocio.

Al comparar las ventas por categoría, Technology lidera el total de ventas, 
seguida de cerca por Office Supplies, mientras que Furniture presenta el menor 
volumen. Sin embargo, la mayoría de los pedidos individuales corresponden a 
montos de venta bajos (la gran mayoría se concentra por debajo de 5,000 en 
ventas por pedido), lo que refleja que el negocio depende de un alto volumen de 
transacciones pequeñas más que de pocas ventas grandes.

La relación entre descuento y ganancia muestra una tendencia clara: a medida que 
el descuento aplicado aumenta (especialmente por encima de 0.3), la ganancia por 
pedido tiende a reducirse e incluso volverse negativa, como se observa en los 
descuentos de 0.5 y 0.7. Esto indica que otorgar descuentos altos está erosionando 
el margen de ganancia, por lo que se recomienda revisar y limitar la política de 
descuentos superiores al 30%.

Use los filtros de Región y Categoría en la parte superior para explorar el 
comportamiento de cada segmento en particular.